# 02 — Model-backed image editing

Uses tool-owned SAM3, ObjectClear, Stable Diffusion, Diffusers GLIGEN, and Flux pipelines. Models load lazily on first use.


In [ ]:
# %pip install -e '.[models]'

In [ ]:
import os
from PIL import Image
from IPython.display import display
from smolagents import OpenAIModel
from core import Agent
from tools import image_editing_tools

model = OpenAIModel(
    model_id='gemini-2.5-flash',
    api_base='https://generativelanguage.googleapis.com/v1beta/openai/',
    api_key=os.environ['GOOGLE_API_KEY'],
    temperature=0,
)


In [ ]:
IMAGE_PATH = 'your_image.jpg'
image = Image.open(IMAGE_PATH).convert('RGB')
display(image)

# Replace with coordinates from your UI.
point_0 = [image.width // 2, image.height // 2]
box_0 = [20, 20, min(image.width - 20, 260), min(image.height - 20, 220)]
scribble_0 = [[100, 100], [120, 120], [140, 130], [160, 145]]

In [ ]:
tools = image_editing_tools()
agent = Agent(model=model, tools=tools, verbosity_level=1)


In [ ]:
result = agent.send(
    'Remove the object selected by <point> and send the result.',
    objects={'image_0': image, 'point_0': point_0},
)
if result.message.objects:
    display(result.message.objects[0])
print(agent.trajectory_text)

## Multi-turn follow-up

The agent keeps its previous image variables. You can add a new visual prompt on the next user turn.

In [ ]:
result2 = agent.send(
    'Now make the object selected by <box> about 1.4x larger and send the image.',
    objects={'box_1': box_0},
)
if result2.message.objects:
    display(result2.message.objects[0])

## Optional: generative inpainting with bundled inference models


In [ ]:
result = agent.send(
    'Add a red sports car at <box> and send the image.',
    objects={'image_0': image, 'box_0': box_0},
)
if result.message.objects:
    display(result.message.objects[0])
